In [ ]:
pip install open_clip_torch

In [ ]:
# Install required libraries if you are running this in Google Colab:
# !pip install open_clip_torch scikit-learn tqdm pandas matplotlib

import os
import json
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from PIL import Image
from tqdm.notebook import tqdm # Using the notebook version of tqdm for prettier progress bars
from sklearn.metrics import accuracy_score, f1_score, classification_report
import open_clip
from dataclasses import dataclass

In [ ]:
@dataclass
class ProjectConfig:
    # 1. File Paths (Ensure these match your notebook's location)
    json_path: str = 'data/dataset_rsicd.json'
    image_dir: str = 'data/RSICD_images'
    txtclasses_dir: str = 'data/txtclasses_rsicd'

    # 2. Model Settings
    model_name: str = 'ViT-B-32'
    pretrained: str = 'laion2b_s34b_b79k'

    # 3. Training Hyperparameters
    batch_size: int = 32
    learning_rate: float = 1e-5
    epochs: int = 10
    weight_decay: float = 0.001

    # 4. Hardware
    device: str = 'cuda' if torch.cuda.is_available() else 'cpu'

# Initialize the config so we can use it in the notebook
config = ProjectConfig()
print(f"Project configured to run on: {config.device}")

In [ ]:
class RSICDDataset(Dataset):
    def __init__(self, json_path, image_dir, txtclasses_dir, split_name, transform=None):
        self.image_dir = image_dir
        self.transform = transform

        # 1. Build the Label Mapping and dynamic class list
        self.label_mapping = {}
        self.classes = []

        if os.path.isdir(txtclasses_dir):
            for txt_file in os.listdir(txtclasses_dir):
                if txt_file.endswith('.txt'):
                    class_name = txt_file.replace('.txt', '').lower()
                    self.classes.append(class_name)

                    with open(os.path.join(txtclasses_dir, txt_file), 'r') as f:
                        for line in f:
                            filename = line.strip()
                            if filename:
                                self.label_mapping[filename] = class_name

        self.classes = sorted(self.classes)

        # 2. Load JSON and filter by split
        with open(json_path, 'r') as f:
            data = json.load(f)

        self.samples = [item for item in data['images'] if item['split'] == split_name]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        item = self.samples[idx]
        filename = item['filename']

        # Load image
        img_path = os.path.join(self.image_dir, filename)
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)

        # Extract captions
        captions = [sentence['raw'] for sentence in item['sentences']] if 'sentences' in item else []

        # Look up correct label
        label = self.label_mapping.get(filename, "unknown")

        return image, label, captions

In [ ]:
class CustomCLIP(nn.Module):
    def __init__(self, model_name='ViT-B-32', pretrained='laion2b_s34b_b79k'):
        super().__init__()

        self.model, _, self.preprocess = open_clip.create_model_and_transforms(
            model_name,
            pretrained=pretrained
        )

        self.tokenizer = open_clip.get_tokenizer(model_name)

    def forward(self, images, text_tokens):
        image_features, text_features, logit_scale = self.model(images, text_tokens)
        return image_features, text_features, logit_scale

    def encode_image(self, images):
        return self.model.encode_image(images)

    def encode_text(self, text_tokens):
        return self.model.encode_text(text_tokens)

In [ ]:
class BaseTrainer:
    def __init__(self, model, dataloader, optimizer, device):
        self.model = model.to(device)
        self.dataloader = dataloader
        self.optimizer = optimizer
        self.device = device
        self.criterion = nn.CrossEntropyLoss()

    def train_epoch(self, epoch):
        self.model.train()
        total_loss = 0.0

        progress_bar = tqdm(self.dataloader, desc=f"Epoch {epoch} Training", leave=False)

        for batch_idx, (images, labels, captions) in enumerate(progress_bar):
            images = images.to(self.device)

            # Unpack the PyTorch caption tuple correctly
            texts = list(captions[0])
            text_tokens = self.model.tokenizer(texts).to(self.device)

            self.optimizer.zero_grad()

            image_features, text_features, logit_scale = self.model(images, text_tokens)

            logits_per_image = logit_scale * image_features @ text_features.T
            logits_per_text = logits_per_image.T

            # Ground truth targets form a diagonal line
            labels = torch.arange(images.shape[0], device=self.device)

            loss_i = self.criterion(logits_per_image, labels)
            loss_t = self.criterion(logits_per_text, labels)
            total_batch_loss = (loss_i + loss_t) / 2

            total_batch_loss.backward()
            self.optimizer.step()

            total_loss += total_batch_loss.item()
            progress_bar.set_postfix({"Loss": f"{total_batch_loss.item():.4f}"})

        return total_loss / len(self.dataloader)

In [ ]:
class Evaluator:
    def __init__(self, model, dataloader, class_names, device):
        self.model = model.to(device)
        self.dataloader = dataloader
        self.class_names = class_names
        self.device = device

    def evaluate_zero_shot(self):
        self.model.eval()
        templates = [f"A satellite image of a {c}" for c in self.class_names]
        text_tokens = self.model.tokenizer(templates).to(self.device)

        all_preds = []
        all_labels = []

        with torch.no_grad():
            text_features = self.model.encode_text(text_tokens)
            text_features /= text_features.norm(dim=-1, keepdim=True)

            for images, labels, captions in tqdm(self.dataloader, desc="Zero-Shot Eval", leave=False):
                images = images.to(self.device)

                label_indices = [self.class_names.index(l) for l in labels]
                all_labels.extend(label_indices)

                image_features = self.model.encode_image(images)
                image_features /= image_features.norm(dim=-1, keepdim=True)

                similarity = (100.0 * image_features @ text_features.T).softmax(dim=-1)
                preds = similarity.argmax(dim=-1).cpu().numpy()
                all_preds.extend(preds)

        accuracy = accuracy_score(all_labels, all_preds)
        macro_f1 = f1_score(all_labels, all_preds, average='macro')
        report = classification_report(all_labels, all_preds, target_names=self.class_names, zero_division=0)

        return accuracy, macro_f1, report

    def evaluate_retrieval(self):
        self.model.eval()
        all_image_features = []
        all_text_features = []

        with torch.no_grad():
            for images, labels, captions in tqdm(self.dataloader, desc="Retrieval Eval", leave=False):
                images = images.to(self.device)

                texts = list(captions[0])
                text_tokens = self.model.tokenizer(texts).to(self.device)

                image_features = self.model.encode_image(images)
                image_features /= image_features.norm(dim=-1, keepdim=True)
                all_image_features.append(image_features)

                text_features = self.model.encode_text(text_tokens)
                text_features /= text_features.norm(dim=-1, keepdim=True)
                all_text_features.append(text_features)

        all_image_features = torch.cat(all_image_features, dim=0)
        all_text_features = torch.cat(all_text_features, dim=0)

        similarity_matrix = (100.0 * all_image_features @ all_text_features.T)

        metrics = {}
        for k in [1, 5, 10]:
            topk_i2t = similarity_matrix.topk(k, dim=1)[1]
            correct_i2t = torch.arange(len(all_image_features)).view(-1, 1).to(self.device)
            r_at_k_i2t = (topk_i2t == correct_i2t).any(dim=1).float().mean().item() * 100
            metrics[f"Image-to-Text R@{k}"] = r_at_k_i2t

            topk_t2i = similarity_matrix.topk(k, dim=0)[1].T
            correct_t2i = torch.arange(len(all_text_features)).view(-1, 1).to(self.device)
            r_at_k_t2i = (topk_t2i == correct_t2i).any(dim=1).float().mean().item() * 100
            metrics[f"Text-to-Image R@{k}"] = r_at_k_t2i

        return metrics

In [ ]:
print(f"--- Initializing RSICD Fine-Tuning ---")

# 1. Initialize Architecture
model = CustomCLIP(model_name=config.model_name, pretrained=config.pretrained)

# 2. Prepare Data Pipelines (Using the 'val' split fix)
train_dataset = RSICDDataset(
    json_path=config.json_path,
    image_dir=config.image_dir,
    txtclasses_dir=config.txtclasses_dir,
    split_name='val',
    transform=model.preprocess
)
train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True, drop_last=True)

test_dataset = RSICDDataset(
    json_path=config.json_path,
    image_dir=config.image_dir,
    txtclasses_dir=config.txtclasses_dir,
    split_name='test',
    transform=model.preprocess
)
test_loader = DataLoader(test_dataset, batch_size=config.batch_size, shuffle=False)

# 3. Setup Optimizer
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=config.learning_rate,
    weight_decay=config.weight_decay
)

# 4. Initialize Engines
trainer = BaseTrainer(model, train_loader, optimizer, config.device)
evaluator = Evaluator(model, test_loader, test_dataset.classes, config.device)

# 5. Baseline Evaluation (Phase 1)
print("\n[ Phase 1: Baseline Evaluation ]")
base_acc, base_f1, _ = evaluator.evaluate_zero_shot()
base_retrieval = evaluator.evaluate_retrieval()

print(f"Baseline Zero-Shot | Acc: {base_acc:.4f} | F1: {base_f1:.4f}")
print(f"Baseline Retrieval | I2T R@1: {base_retrieval['Image-to-Text R@1']:.2f}% | T2I R@1: {base_retrieval['Text-to-Image R@1']:.2f}%")

# 6. The Training Loop (Phase 2)
print("\n[ Phase 2: Contrastive Fine-Tuning ]")
best_accuracy = 0.0

for epoch in range(1, config.epochs + 1):
    # Train
    avg_train_loss = trainer.train_epoch(epoch)

    # Evaluate
    val_acc, val_f1, _ = evaluator.evaluate_zero_shot()
    retrieval_metrics = evaluator.evaluate_retrieval()

    # Print Dashboard
    print(f"\n--- Epoch {epoch} Summary ---")
    print(f"Train Loss:     {avg_train_loss:.4f}")
    print(f"Classification: Acc: {val_acc:.4f} | F1: {val_f1:.4f}")
    print(f"Retrieval:      I2T R@1: {retrieval_metrics['Image-to-Text R@1']:.2f}% | T2I R@1: {retrieval_metrics['Text-to-Image R@1']:.2f}%")

    # Checkpoint
    if val_acc > best_accuracy:
        best_accuracy = val_acc
        print(">>> Peak Accuracy Reached! Saving best_clip_model.pth <<<")
        torch.save(model.state_dict(), "best_clip_model.pth")

print("\n--- Training Complete ---")